This script inputs the structure of a protein in PDB format and calculates the contact matrix that is used for the allosteric pathway-finding algorithm described in Wang et al - https://doi.org/10.1038/s41467-020-17618-2.

The contact matrix is stored as 'proteinname_contact_coupling.csv' without header and index

In [1]:
!pip install biopython numpy scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from Bio.PDB import PDBParser

In [3]:
Protein_name = 'Ras'
PDB_PATH      = f"{Protein_name}.pdb"
CHAIN_IDS     = None
CONTACT_CUTOFF = 3.4
SAVE_CSV      = True
OUTPUT_CSV    = f"{Protein_name}_contact_coupling.csv"

In [4]:
BACKBONE = {"N", "CA", "C", "O"}

parser = PDBParser(QUIET=True)
model  = next(parser.get_structure("prot", PDB_PATH).get_models())

res_coords  = []
res_bb_idx  = []
labels      = []
ca_coords   = []

for chain in model:
    if CHAIN_IDS and chain.id not in CHAIN_IDS:
        continue
    for res in chain:
        het, seqid, _ = res.get_id()
        if het.strip():
            continue
        atoms = list(res.get_atoms())
        if not atoms:
            continue
        coords  = np.array([a.get_vector().get_array() for a in atoms])
        bb_set  = {k for k, a in enumerate(atoms) if a.name in BACKBONE}
        ca_atom = next((a for a in atoms if a.name == "CA"), atoms[0])
        res_coords.append(coords)
        res_bb_idx.append(bb_set)
        labels.append(f"{chain.id}/{seqid}")
        ca_coords.append(ca_atom.get_vector().get_array())

atom_counts = np.array([len(c) for c in res_coords], dtype=np.int32)
ca_coords   = np.array(ca_coords)
N_res       = len(labels)

print(f"Parsed {N_res} residues from {PDB_PATH}")

Parsed 87 residues from Ras.pdb


In [5]:
def build_contact_matrix(residue_atoms, backbone_idx, cutoff=CONTACT_CUTOFF):
    N, cutoff_sq = len(residue_atoms), cutoff ** 2
    C = np.zeros((N, N), dtype=np.int32)
    for i in range(N):
        ci = residue_atoms[i]
        for j in range(i + 1, N):
            cj   = residue_atoms[j]
            diff = ci[:, None, :] - cj[None, :, :]
            mask = (diff * diff).sum(axis=2) <= cutoff_sq
            if abs(i - j) == 1:
                bb_i = np.array(sorted(backbone_idx[i]), dtype=np.intp)
                bb_j = np.array(sorted(backbone_idx[j]), dtype=np.intp)
                if bb_i.size and bb_j.size:
                    mask[np.ix_(bb_i, bb_j)] = False
            C[i, j] = C[j, i] = int(mask.sum())
    return C


C = build_contact_matrix(res_coords, res_bb_idx, CONTACT_CUTOFF)
print(f"Done. C shape: {C.shape}, max contacts: {C.max()}")

Done. C shape: (87, 87), max contacts: 5


In [6]:
eps   = 1e-12
N_fwd = C / (atom_counts[:, None].astype(float) + eps)
N_rev = C / (atom_counts[None, :].astype(float) + eps)

In [8]:
import pandas as pd
pd.DataFrame(N_fwd).to_csv(OUTPUT_CSV, index = False, header = False)